# Box-score-only value metric

Fits team-level per-100 box rates against team NRtg via RidgeCV, independent of RAPM, then applies the fitted coefficients to each player's own per-100 rates. Run `pull_traded_stints.ipynb` and `pull_positions.ipynb` first.

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.stats import zscore
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_score

HERE = os.getcwd()
BASE = os.path.dirname(HERE)
DATA_DIR = os.path.join(HERE, "data")
PS_PATH = os.path.join(BASE, "shinyrapmapp", "data", "playerstats_all.csv")
STINTS_PATH = os.path.join(HERE, "traded_player_stints.csv")
POSITIONS_PATH = os.path.join(HERE, "player_positions.csv")

MIN_THRESHOLD = 250  # players below this are too small a sample for per-100 rates to be meaningful

SEASONS = [
    ("1718", 2017), ("1819", 2018), ("1920", 2019), ("2021", 2020),
    ("2122", 2021), ("2223", 2022), ("2324", 2023), ("2425", 2024), ("2526", 2025),
]
full_to_bbref = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BRK",
    "Charlotte Hornets": "CHO", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHO",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS",
}
bbref_to_nba = {v: v for v in full_to_bbref.values()}
bbref_to_nba.update({"BRK": "BKN", "CHO": "CHA", "PHO": "PHX"})

box_cols = ["FG","FGA","3P","3PA","FT","FTA","ORB","DRB","AST","STL","BLK","TOV","PF"]
per100_cols = [f"{c}_100" for c in box_cols]

## Team-level: assemble per-100 box rates + NRtg, fit RidgeCV

In [ ]:
team_rows = []
for suffix, label in SEASONS:
    tt = pd.read_csv(os.path.join(DATA_DIR, f"teamtotals{suffix}.csv"))
    tt["bbref"] = tt["Team"].map(full_to_bbref)
    assert tt["bbref"].isna().sum() == 0, f"unmapped team in {suffix}"
    tt["EstPoss"] = tt["FGA"] - tt["ORB"] + tt["TOV"] + 0.44 * tt["FTA"]
    for c in box_cols:
        tt[f"{c}_100"] = tt[c] / tt["EstPoss"] * 100

    ts = pd.read_csv(os.path.join(DATA_DIR, f"teamstats{suffix}.csv"))
    ts.columns = [c.strip() for c in ts.columns]
    ts["Tm"] = ts["Tm"].astype(str).str.strip()
    ts["NRtg_clean"] = ts["NRtg"].astype(str).str.replace("+", "", regex=False).astype(float)

    merged = tt.merge(ts[["Tm", "NRtg_clean"]], left_on="bbref", right_on="Tm", how="left")
    assert merged["NRtg_clean"].isna().sum() == 0, f"unmatched NRtg in {suffix}"
    merged["Season"] = label
    merged["nba_abbrev"] = merged["bbref"].map(bbref_to_nba)
    team_rows.append(merged[["bbref", "nba_abbrev", "Season", "EstPoss", "MP"] + per100_cols + ["NRtg_clean"]])

team_data = pd.concat(team_rows, ignore_index=True)
print(f"Assembled {len(team_data)} team-seasons")

X = team_data[per100_cols].values
y = team_data["NRtg_clean"].values
ridge = RidgeCV(alphas=np.logspace(-2, 4, 200))
ridge.fit(X, y)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(ridge, X, y, cv=kf, scoring="r2")
print(f"RidgeCV chosen alpha: {ridge.alpha_:.3f}")
print(f"CV R2: {cv_r2.mean():.4f} (+/- {cv_r2.std():.4f})")

coef_table = pd.DataFrame({"stat": box_cols, "coef": ridge.coef_})
print(f"\nintercept: {ridge.intercept_:.4f}")
coef_table

OLS, RidgeCV, and ElasticNetCV all land within noise of each other (CV R² ~0.857-0.859); RidgeCV is used for consistency with the rest of the project.

## Individual players: apply the model, with trade-stint correction

In [ ]:
ps = pd.read_csv(PS_PATH)
stints = pd.read_csv(STINTS_PATH)

traded_keys = set(zip(stints["PLAYER_ID"], stints["Season"]))
ps_single = ps[~ps.apply(lambda r: (r["PLAYER_ID"], r["Season"]) in traded_keys, axis=1)].copy()
print(f"playerstats_all rows: {len(ps)}, removed {len(ps) - len(ps_single)} mislabeled traded-player rows, kept {len(ps_single)}")
print(f"replacing with {len(stints)} correct stint rows -> combined total: {len(ps_single) + len(stints)}")

raw_to_per100 = {
    "FG": "FGM", "FGA": "FGA", "3P": "FG3M", "3PA": "FG3A", "FT": "FTM", "FTA": "FTA",
    "ORB": "OREB", "DRB": "DREB", "AST": "AST", "STL": "STL", "BLK": "BLK", "TOV": "TOV", "PF": "PF",
}
keep_cols = ["PLAYER_ID","PLAYER_NAME","TEAM_ABBREVIATION","Season","MIN"] + list(raw_to_per100.values())
combined = pd.concat([ps_single[keep_cols], stints[keep_cols]], ignore_index=True)

team_lookup = team_data.set_index(["nba_abbrev", "Season"])[["EstPoss", "MP"]]
combined = combined.join(team_lookup, on=["TEAM_ABBREVIATION", "Season"])
missing = combined["EstPoss"].isna().sum()
print(f"rows missing a team match: {missing} / {len(combined)}")
combined = combined.dropna(subset=["EstPoss", "MP"]).copy()

combined["PlayerPoss"] = combined["EstPoss"] * (combined["MIN"] / combined["MP"])
for stat, src_col in raw_to_per100.items():
    combined[f"{stat}_100"] = combined[src_col] / combined["PlayerPoss"] * 100

X_player = combined[per100_cols].values
# raw = predicted NRtg for a hypothetical team of 5 clones of this player;
# /5 converts that team-scale number into each clone's individual share
combined["BoxOnlyValue"] = (ridge.intercept_ + X_player @ ridge.coef_) / 5
print("BoxOnlyValue computed (raw team-calibrated scale -- see note below on why this isn't the final reported number).")

## Why the final numbers are percentiles, not raw point-differential values

Two alternatives (refitting per-player against team NRtg, and winsorizing inputs) were tried and rejected because they either collapsed out-of-sample fit or erased what makes specialists distinguishable. The team-level fit's raw ranking is sound, so it's reported as a percentile against that season's population instead of a literal point-differential number.

In [ ]:
def add_standardized(df, value_col, group_cols, min_col="MIN", threshold=MIN_THRESHOLD):
    df = df.copy()
    df[f"{value_col}_z"] = np.nan
    df[f"{value_col}_pct"] = np.nan
    qualified = df[df[min_col] >= threshold]
    for _, idx in qualified.groupby(group_cols).groups.items():
        if len(idx) < 2:
            continue
        sub = qualified.loc[idx]
        df.loc[idx, f"{value_col}_z"] = zscore(sub[value_col])
        df.loc[idx, f"{value_col}_pct"] = sub[value_col].rank(pct=True) * 100
    return df

combined = add_standardized(combined, "BoxOnlyValue", group_cols=["Season"])
print("Added BoxOnlyValue_z / BoxOnlyValue_pct (within-season, 250+min population).")

## Position-relative value

Reports value relative to positional peers as well, since a single universal ranking favors big, high-rebounding players whose production is more easily counted in box stats.

In [ ]:
if not os.path.exists(POSITIONS_PATH):
    print("player_positions.csv not found -- run pull_positions.ipynb first. Skipping positional adjustment.")
    combined["PosBucket"] = np.nan
    combined["ValueAbovePosition"] = np.nan
else:
    positions = pd.read_csv(POSITIONS_PATH)
    combined = combined.merge(positions, on=["PLAYER_ID", "Season"], how="left")
    missing_pos = combined["PosBucket"].isna().sum()
    print(f"rows missing a position match: {missing_pos} / {len(combined)}")

    pos_pop = combined[combined["MIN"] >= MIN_THRESHOLD].dropna(subset=["PosBucket"])
    pos_avg = pos_pop.groupby(["Season", "PosBucket"]).apply(
        lambda g: (g["BoxOnlyValue"] * g["MIN"]).sum() / g["MIN"].sum(), include_groups=False
    ).rename("PosAvgValue").reset_index()
    combined = combined.merge(pos_avg, on=["Season", "PosBucket"], how="left")
    combined["ValueAbovePosition"] = combined["BoxOnlyValue"] - combined["PosAvgValue"]
    combined = add_standardized(combined, "ValueAbovePosition", group_cols=["Season", "PosBucket"])
    print("Added ValueAbovePosition, ValueAbovePosition_z, ValueAbovePosition_pct.")

out_path = os.path.join(DATA_DIR, "box_only_metric.csv")
combined.sort_values(["Season", "BoxOnlyValue"], ascending=[True, False]).to_csv(out_path, index=False)
print(f"Wrote {out_path}")

leaderboard = combined[combined["MIN"] >= MIN_THRESHOLD].sort_values(["Season", "BoxOnlyValue_pct"], ascending=[True, False])
leaderboard_cols = ["PLAYER_NAME","TEAM_ABBREVIATION","Season","PosBucket","MIN",
                     "BoxOnlyValue","BoxOnlyValue_pct","ValueAbovePosition","ValueAbovePosition_pct"]
leaderboard_path = os.path.join(DATA_DIR, "box_only_leaderboard.csv")
leaderboard[leaderboard_cols].round(2).to_csv(leaderboard_path, index=False)
print(f"Wrote {leaderboard_path} ({len(leaderboard)} rows, {MIN_THRESHOLD}+ min floor)")

## Sanity checks

In [ ]:
# internal consistency: minutes-weighted player predictions, reaggregated to team level,
# should closely track real team NRtg.
agg = combined.groupby(["TEAM_ABBREVIATION", "Season"]).apply(
    lambda g: pd.Series({"WeightedBoxValue": (g["BoxOnlyValue"] * g["MIN"]).sum() / g["MIN"].sum()}),
    include_groups=False,
).reset_index()
cmp = agg.merge(team_data[["nba_abbrev", "Season", "NRtg_clean"]],
                 left_on=["TEAM_ABBREVIATION", "Season"], right_on=["nba_abbrev", "Season"])
print(f"corr(team-reaggregated predictions, real NRtg): {cmp['WeightedBoxValue'].corr(cmp['NRtg_clean']):.4f}")
print(f"(team-level model's own CV correlation ceiling is ~{cv_r2.mean() ** 0.5:.4f})")

print("\nTop 10 by BoxOnlyValue_pct, most recent season (250+ min):")
recent = combined[(combined["Season"] == combined["Season"].max()) & (combined["MIN"] >= MIN_THRESHOLD)]
recent.sort_values("BoxOnlyValue_pct", ascending=False).head(10)[
    ["PLAYER_NAME","TEAM_ABBREVIATION","PosBucket","MIN","BoxOnlyValue_pct","ValueAbovePosition_pct"]
]